# 01 -- Data and Source Validation

Bock & Maewal (2023) paper rebuild. Validates the paper's own 11-ETF universe against real Yahoo Finance data, builds the weekly calendar, and audits data quality. No strategy tuning happens in this notebook -- see `docs/paper_source_audit.md` for full source evidence.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data import PAPER_UNIVERSE, VANGUARD_UNIVERSE_FORBIDDEN, assert_canonical_order, assert_no_vanguard_substitution, load_universe_prices
from src.calendar import build_weekly_calendar, weeks_with_unavailable_target
from src.labels import build_open_close_panels, build_paper_labels
from src.tensors import build_weekly_price_matrix
from src.normalization import fit_annual_price_scaler

print('Canonical universe (Table 1 order):', PAPER_UNIVERSE)

Canonical universe (Table 1 order): ('XLK', 'XLV', 'XLY', 'VOX', 'XLF', 'XLI', 'XLP', 'XLU', 'XLB', 'IYR', 'XLE')


## Universe verification

Assert the paper's own 11 tickers are used, and that none of the archived project's Vanguard-fund tickers have leaked in.

In [2]:
assert_canonical_order(PAPER_UNIVERSE)
assert_no_vanguard_substitution(PAPER_UNIVERSE)
print('Universe OK: 11 tickers, Table-1 order, no Vanguard substitution.')
print('Forbidden (archived-project) tickers, confirmed absent:', sorted(VANGUARD_UNIVERSE_FORBIDDEN))

Universe OK: 11 tickers, Table-1 order, no Vanguard substitution.
Forbidden (archived-project) tickers, confirmed absent: ['VAW', 'VCR', 'VDC', 'VDE', 'VFH', 'VGT', 'VHT', 'VIS', 'VNQ', 'VPU']


## Real data load (Yahoo Finance, immutable raw CSVs under `data/raw/prices/`)

In [3]:
prices = load_universe_prices()
for symbol, df in prices.items():
    print(f"{symbol:5s}  rows={len(df):5d}  {df['date'].min().date()} -> {df['date'].max().date()}")

XLK    rows= 6952  1998-12-22 -> 2026-08-13
XLV    rows= 6952  1998-12-22 -> 2026-08-13
XLY    rows= 6952  1998-12-22 -> 2026-08-13
VOX    rows= 5503  2004-09-29 -> 2026-08-13
XLF    rows= 6952  1998-12-22 -> 2026-08-13
XLI    rows= 6952  1998-12-22 -> 2026-08-13
XLP    rows= 6952  1998-12-22 -> 2026-08-13
XLU    rows= 6952  1998-12-22 -> 2026-08-13
XLB    rows= 6952  1998-12-22 -> 2026-08-13
IYR    rows= 6577  2000-06-19 -> 2026-08-13
XLE    rows= 6952  1998-12-22 -> 2026-08-13


## Data audit

See `outputs/paper_data_audit.csv` for the full per-symbol gap/null audit (already generated; reproduced here for a quick visual check). All 11 series cover the paper's 2012-2022 window with 2 years of prior history (>= 2010) and no data gaps beyond ordinary weekends/holidays.

In [4]:
audit_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'paper_data_audit.csv')
audit_df[['symbol', 'history_start', 'history_end', 'gaps_over_5_days_2010_2022', 'covers_full_paper_period_2012_2022']]

,symbol,history_start,history_end,gaps_over_5_days_2010_2022,covers_full_paper_period_2012_2022
0,GSPC,1927-12-30,2026-08-13,0,True
1,IYR,2000-06-19,2026-08-13,0,True
2,SPY,1993-01-29,2026-08-13,0,True
3,VOX,2004-09-29,2026-08-13,0,True
4,XLB,1998-12-22,2026-08-13,0,True
5,XLE,1998-12-22,2026-08-13,0,True
6,XLF,1998-12-22,2026-08-13,0,True
7,XLI,1998-12-22,2026-08-13,0,True
8,XLK,1998-12-22,2026-08-13,0,True
9,XLP,1998-12-22,2026-08-13,0,True


## Weekly calendar reconstruction

Built from real observed trading days (not a synthetic holiday calendar) -- see `docs/paper_execution_timeline.md`. `DECISION_REQUIRED_HOLIDAY_EXECUTION` is **RESOLVED**: a week's entry/exit sessions are its first/last *actual* trading day, so a missing literal Friday close or Monday open no longer blocks execution -- demonstrated below on a real holiday-shortened week (Good Friday 2021).

In [5]:
spy = prices['XLK']  # any universe member works; trading-day index is shared across US equities
trading_days = pd.DatetimeIndex(spy[(spy['date'] >= '2021-01-01') & (spy['date'] <= '2021-12-31')]['date'])
calendar_2021 = build_weekly_calendar(trading_days)
calendar_2021[calendar_2021['friday_present'] == False]

,week_id,first_actual_trading_day,last_actual_trading_day,friday_present,model_cutoff,entry_candidate_date,exit_candidate_date,label_known_date
12,12,2021-03-29,2021-04-01,False,2021-04-01,2021-04-05,2021-04-01,2021-04-09
50,50,2021-12-20,2021-12-23,False,2021-12-23,2021-12-27,2021-12-23,2021-12-31


In [6]:
open_panel, close_panel = build_open_close_panels(prices)
labels_2021 = build_paper_labels(calendar_2021, open_panel, close_panel)

# week_id 11's target is week_id 12 -- the Good-Friday-shortened week above
# (first_actual_trading_day 2021-03-29 Mon, last_actual_trading_day 2021-04-01 Thu).
# No PaperDecisionRequiredError: DECISION_REQUIRED_HOLIDAY_EXECUTION is resolved.
holiday_target_row = labels_2021[labels_2021['target_entry_date'] == pd.Timestamp('2021-03-29')]
holiday_target_row[['target_week', 'target_entry_date', 'target_exit_date', 'XLK_target_trade_return', 'XLK_target']]

,target_week,target_entry_date,target_exit_date,XLK_target_trade_return,XLK_target
11,12,2021-03-29,2021-04-01,0.024268,1


## Model input price field (RESOLVED: Adjusted Close levels)

`DECISION_REQUIRED_PRICE_FIELD` is **RESOLVED**: the model's weekly input is Yahoo **Adjusted Close**, as a price **LEVEL** (not raw Close, not a return/log-return/rebased transform), sampled at each week's final *actual* trading session -- reusing the same holiday-aware calendar machinery as the target (`src/tensors.py::build_weekly_price_matrix`). This is a USER-RESOLVED reconstruction decision, not paper-explicit, and applies only to the model input -- the target/execution interval above continues to use RAW Open/Close.

In [7]:
weekly_2021 = build_weekly_price_matrix(prices, calendar_2021)  # default price_field='adjusted_close'

# Raw Close vs Adjusted Close distinction, for a ticker/week where a
# distribution occurred (Close != Adjusted Close):
sample_week = calendar_2021.iloc[0]
sample_date = sample_week['model_cutoff']
raw_close = prices['XLK'].set_index('date').loc[sample_date, 'close']
adjusted_close = prices['XLK'].set_index('date').loc[sample_date, 'adjusted_close']
print(f"XLK on {sample_date.date()}: raw close={raw_close:.4f}  adjusted close={adjusted_close:.4f}")
print('Model input uses adjusted close (a LEVEL, not a return):', weekly_2021.loc[sample_date, 'XLK'] == adjusted_close)

# Weekly sampling on the same Good-Friday-shortened week demonstrated above
# (week_id 12: first/last actual trading day 2021-03-29 Mon / 2021-04-01 Thu):
weekly_2021.loc[[pd.Timestamp('2021-04-01')]]

XLK on 2021-01-08: raw close=65.3800  adjusted close=62.7237
Model input uses adjusted close (a LEVEL, not a return): True


,XLK,XLV,XLY,VOX,XLF,XLI,XLP,XLU,XLB,IYR,XLE
model_cutoff,,,,,,,,,,,
2021-04-01,65.125046,106.914185,81.172356,125.518585,31.483355,91.305801,59.186554,27.309196,35.841061,81.955429,20.758192


## Annual per-ETF z-score scaler (RESOLVED: frozen, initial two-year training history)

`DECISION_REQUIRED_NORMALIZATION_SCOPE` is **RESOLVED**: one z-score scaler per ETF (column-wise, never pooled across tickers), fit once on the annual trading model's initial two-year training history, then frozen for the whole trading year -- `src/normalization.py::AnnualPriceScaler` / `fit_annual_price_scaler`. See `outputs/paper_annual_scaler_audit.csv` for the full per-year, per-ticker audit (all 11 paper trading years, real data).

In [8]:
full_trading_days = pd.DatetimeIndex(prices['XLK']['date'])
calendar_full = build_weekly_calendar(full_trading_days)
weekly_full = build_weekly_price_matrix(prices, calendar_full)

scaler_2015 = fit_annual_price_scaler(weekly_full, trading_year=2015)
print(f"2015 annual scaler: training window [{scaler_2015.training_start.date()}, {scaler_2015.training_end.date()}), "
      f"{scaler_2015.n_training_weeks} weekly observations, ddof={scaler_2015.ddof}")
pd.DataFrame(scaler_2015.audit_rows()).set_index('ticker')[['mean', 'std']]

2015 annual scaler: training window [2013-01-01, 2015-01-01), 104 weekly observations, ddof=0


,mean,std
ticker,,
XLK,14.999364,1.801411
XLV,44.976874,6.504472
XLY,26.954440,2.794848
VOX,66.253140,4.976928
XLF,13.604477,1.393069
XLI,39.133690,4.599119
XLP,30.458911,2.534265
XLU,13.414751,1.210300
XLB,17.329986,1.793454


In [9]:
# One normalized (frozen-scaler) weekly observation, mid-2015 -- inside the
# 2015 trading year, so the *same* frozen 2013-2014 scaler is used, not a
# refit on 2015 data itself:
mid_2015_row = weekly_full.loc[(weekly_full.index >= '2015-06-01') & (weekly_full.index < '2015-06-15')].iloc[[0]]
normalized_row = scaler_2015.transform(mid_2015_row)
print('Raw weekly Adjusted Close levels:', mid_2015_row.iloc[0].to_numpy())
print('Frozen-scaler normalized (z-score):', normalized_row[0])

Raw weekly Adjusted Close levels: [18.87496948 61.70578003 33.84247589 71.42821503 16.31801605 45.7771492
 35.27307129 14.89048004 19.95235634 52.08972168 24.94022179]
Frozen-scaler normalized (z-score): [ 2.15142817  2.57190863  2.46454779  1.03981316  1.94788528  1.44450687
  1.89962759  1.2193087   1.46219002  1.65726072 -0.79052607]


## Summary

Universe, real data acquisition, weekly-calendar mechanics, target-label construction, model input price field, and the annual normalization scaler are all source-supported and implemented. `DECISION_REQUIRED_TARGET_RETURN_INTERVAL`, `DECISION_REQUIRED_HOLIDAY_EXECUTION`, `DECISION_REQUIRED_PRICE_FIELD`, and `DECISION_REQUIRED_NORMALIZATION_SCOPE` are all **RESOLVED** -- see the decision register for the full resolution records and the remaining open list. `DECISION_REQUIRED_LOOKBACK_N` and `DECISION_REQUIRED_VOLUME_INPUT` remain open and are untouched by this notebook.